# Integrated Intelligent Systems: Tutorial 2 - Body Motion Problem (Action Designator & URDF) (3 Pts)

This notebook introduces the body motion problem central to designing and implementing Cognition-Enabled Robot Agents.

## Part 1: Action Designator (E1–E6) (1.5 Pt)

**Title:** The Body Motion Problem: Serving Eva a Drink & Breakfast

![Alternative Text](bmp.png)

### Scenario Background
Eva says: "Serve me a drink."

The kitchen contains two milk cartons in the fridge door:
- **Carton A:** Opened whole milk.
- **Carton B:** Unopened oat drink.

Eva's usual mug already contains hot black coffee on the counter.

A wet surface on Carton A causes slip during a low test lift, requiring a grasp adaptation before pouring.

---

### Multi-Question Student Task Sheet (0.25 Pt x 6)

#### 1. **E1 · The Gap & Whole-Course Judgment**
Eva says "Serve me a drink." A video shows a robot executing a smooth trajectory, placing the mug on the table, but a photograph reveals milk spilled across the counter and an open fridge left behind. Explain why this nominal end-state success does **not** constitute a solution to the Body Motion Problem, and specify two whole-course constraints that were violated.

**Answer:**
- **Why nominal end-state success fails:** A photograph showing the mug on the table only checks the final end state ($t = t_{end}$). However, the Body Motion Problem requires finding and executing motions whose *whole course* accomplishes the intended effect while respecting all task constraints. A task is not solved if the robot violated safety or cleanliness constraints along the way—a successful end state does not excuse damages or hazards caused during execution.
- **Two whole-course constraints violated:**
  1. *Spill prevention / Cleanliness constraint:* Milk was spilled across the counter during transport.
  2. *Environment restoration constraint:* The refrigerator door was left open instead of being closed after getting the milk.

---

#### 2. **E2 · Constrained Action Designator & Grounding**
Distinguish interpretation, inferred means, and grounded objects for this episode by filling out the three missing slots in the schematic action designator:

**Answer:**
- **Working Interpretation:** `(:type serving) (:recipient Eva) (:substance coffee-with-milk)`  
  *(Eva confirmed she wants coffee with milk; this specifies the intended beverage at the task level without yet choosing a specific carton or mug).*
- **Inferred Function/Means:** `(:vessel-role liquid-containment-and-support)`  
  *(To carry and place liquid conveniently, the robot infers that it needs containment and support, creating a candidate vessel role).*
- **Grounded Physical Object:** `(:object Eva-mug-on-counter)`  
  *(Perception connects this inferred vessel role to the physical mug already sitting on the counter with black coffee).* 

---

#### 3. **E3 · The Commitment Decision & Asymmetric Failure Costs**
During the retrieval step, the robot faces a choice between a **5 mm test lift near the shelf** or a **high carry directly to the counter**. State the predicted effect, the cost of failure, and the single piece of evidence that justifies choosing the low test lift before committing to the carry.

**Answer:**
- **Predicted Effect:** The 5 mm test lift checks whether the grip force is sufficient to hold the carton's weight against gravity without slipping. If friction is too low, slip happens immediately over a minimal distance.
- **Cost of Failure:** The costs are strongly asymmetric:
  - *5 mm Test Lift:* Very low / negligible cost. If the carton slips, it only drops 5 mm back onto the fridge shelf. Nothing breaks, nothing spills, and we can easily try another grasp.
  - *High Carry:* Catastrophic cost. If it slips mid-air at carrying height, the carton falls onto the counter or floor, bursts open, spills milk everywhere, and ruins the whole task.
- **Justifying Evidence:** Condensation / wet surface on Carton A, which makes contact friction uncertain.

---

#### 4. **E4 · Causal Prediction across Material Variants**
Compare pouring milk into coffee versus pancake batter into a pan. Formulate the stopping rule for each liquid using `(:stop-when ...)`, and explain how material dynamics alter when the robot must command the wrist to tilt back.

**Answer:**
- **Stopping Rules:**
  - *Milk into coffee:*
    `(:stop-when (>= poured-volume (- target-volume in-flight-volume)))`
  - *Pancake batter into pan:*
    `(:stop-when (>= disc-diameter (- target-diameter spread-margin)))`
- **How material dynamics alter wrist tilt-back:**
  In both cases, issuing the stop command exactly when the target is reached will cause an overshoot, but for different physical reasons:
  - *Milk:* It has low viscosity and pours in a continuous free-falling stream. The milk that already left the carton spout is still in flight and will land in the mug after the pour stops. The robot must tilt the carton back early by this in-flight volume.
  - *Pancake batter:* It is thick and viscous. When poured onto the pan, it pools at the center and keeps spreading outwards even after the stream stops. The robot must stop pouring while the puddle is still smaller than the target disc, letting the spread finish the shape.

---

#### 5. **E5 · System Decomposition & The Reasoning Loop**
Trace the carton slip event through the seven-stage reasoning loop: **Infer → Ground → Predict → Decide → Regulate → Detect & Revise → Verify**. State precisely which step was triggered when downward motion was observed during the 5 mm lift, and how the action designator was developed in response.

**Answer:**
- **Step Triggered:** **`Detect & Revise`** (triggered during execution regulation when sensors detect downward slip, which contradicts the predicted stable hold).
- **Action Designator Development:**
  1. The grasp parameter in the designator was revised from `(:grasp side-pinch)` to `(:grasp bottom-support)` to securely hold the carton from underneath.
  2. Because bottom support blocks the wrist rotation needed for pouring, the designator was further developed to include an intermediate transition: `(:action (:type regrasp) (:location counter) (:before pouring))`, scheduling a regrasp on the counter before attempting to pour.

---

#### 6. **E6 · Conceptual Ablation & Evidence of Competence**
Perform an ablation experiment. Suppose we remove the **"Check before commitment"** capability and replace it with a fixed default high-side-pinch lift. Predict a specific kitchen environment variant where this ablated system fails, and explain why single-run execution success differs from evidence of general competence across task variants.

**Answer:**
- **Predicted Failure Variant:** A kitchen scenario where the milk carton is slippery (e.g. wet surface from condensation like Carton A) or heavier than expected. The ablated robot blindly lifts high with a pinch grasp; the carton slips out at full height, crashes onto the floor, and bursts.
- **Single-run Success vs. General Competence:** A single successful run only shows that the robot didn't fail under that one specific situation (which could just be a "lucky success" if the carton happened to be dry or light). General competence means the robot's reasoning, perceptual safeguards, and recovery strategies work reliably across different conditions (varying weights, slippery surfaces, different carton shapes, or unexpected disturbances).

## Part 2: Kino-Dynamic Modeling of Robot World System with URDF (1.5 Pt)

**Title:**  Designing and Implementing the kino-dynamic model of the robot world system with URDF

### 2.1  Theoretical Background

In [1]:
from IPython.display import display, HTML, IFrame

# Direct HTML rendering for seamless VS Code & JupyterLab display
try:
    with open('urdf_background.html', 'r', encoding='utf-8') as f:
        display(HTML(f.read()))
except Exception:
    display(IFrame('urdf_background.html', width=1000, height=1200))


## 2.2  Hands-on: Model your own robot world system according to specifications below (1.5 Pt)

In [2]:
from IPython.display import display, HTML, IFrame

# Direct HTML rendering for seamless VS Code & JupyterLab display
try:
    with open('urdf_handon.html', 'r', encoding='utf-8') as f:
        display(HTML(f.read()))
except Exception:
    display(IFrame('urdf_handon.html', width=1000, height=1400))


### 2.2.1 Robot Design & Implementation Notes: The Domestic Assistant Robot

For the hands-on challenge, I designed a modular robot model in Xacro to handle the drink-serving task according to the environmental specifications:

#### 1. Component Structure
- **`base.xacro` (Base pedestal):** A cylindrical pedestal with parameterized radius (`${base_radius} = 0.25 m`) and height (`${base_height} = 0.30 m`). It provides a stable base on the floor anchored to the `world` link.
- **`arm.xacro` (Manipulator Arm):** A 3-DOF arm with parameterized link lengths (`${link1_length} = 0.45 m`, `${link2_length} = 0.40 m`).
  - `shoulder_pitch_joint`: Revolute around Y ($-1.57$ to $+1.57$ rad) to pitch up and down.
  - `elbow_pitch_joint`: Revolute around Y ($-2.0$ to $+2.0$ rad) for extending/retracting reach.
  - `wrist_roll_joint`: Revolute around Z ($-3.14$ to $+3.14$ rad) to provide wrist rotation for pouring milk.
- **`gripper.xacro` (Parallel Gripper):** A 2-finger parallel gripper with parameterized span (`${gripper_span} = 0.14 m`). Two prismatic joints open up to 0.07 m each from the center. Since the milk carton is 0.10 m wide, this gives 4 cm of clearance to grasp it easily.
- **`robot.xacro`:** Top-level assembly file that includes the sub-files, sets global properties, and connects the base, arm, and gripper together.
- **`robot.urdf`:** Compiled standalone URDF file ready to load in simulation (RViz/Gazebo).

#### 2. Reachability Check
- **Reaching into fridge shelf (0.40 m):** With the base height at 0.30 m, pitching the shoulder downward lets the gripper reach down to 0.40 m (and down to 0.10 m minimum) to grasp the carton cleanly.
- **Reaching above the counter to pour into Eva's mug (1.00 m - 1.05 m):** The counter is at 0.90 m and Eva's mug is 0.12 m tall. Raising the shoulder and extending the arm allows the end-effector to reach up to 1.02 m - 1.27 m, safely clearing the counter to pour.
- **Gripper clearance:** The 0.14 m span comfortably encloses the 0.10 m milk carton.

In [3]:
# ====================================================================
# URDF Verification Script: Validating Kinematics, Links, and Limits
# ====================================================================
import xml.etree.ElementTree as ET

tree = ET.parse('robot.urdf')
root = tree.getroot()

links = [link.attrib['name'] for link in root.findall('link')]
joints = root.findall('joint')

print("=" * 70)
print(f"URDF Verification: {root.attrib.get('name')}")
print("=" * 70)
print(f"Total Links: {len(links)} -> {links}")
print(f"Total Joints: {len(joints)}")

for j in joints:
    j_name = j.attrib['name']
    j_type = j.attrib['type']
    parent = j.find('parent').attrib['link']
    child = j.find('child').attrib['link']
    origin = j.find('origin').attrib if j.find('origin') is not None else {}
    limit = j.find('limit').attrib if j.find('limit') is not None else {}
    xyz = origin.get('xyz', '0 0 0')
    if j_type == 'prismatic':
        print(f"  [{j_type:9}] {j_name}: {parent} -> {child} | span: [{limit.get('lower')}, {limit.get('upper')}] m")
    elif j_type == 'revolute':
        print(f"  [{j_type:9}] {j_name}: {parent} -> {child} @ xyz='{xyz}' | limits: [{limit.get('lower')}, {limit.get('upper')}] rad, vel={limit.get('velocity')} m/s")
    else:
        print(f"  [{j_type:9}] {j_name}: {parent} -> {child} @ xyz='{xyz}'")

# Reachability and Dimension Verification
base_z = 0.30
l1 = 0.45
l2 = 0.40
wrist_to_tip = 0.12
max_vertical_reach = base_z + l1 + l2 + wrist_to_tip
min_vertical_reach = max(0.1, base_z - (l1 + l2 - wrist_to_tip) * 0.7)

print("\n--- Reach & Geometry Validation ---")
print(f"✓ Base Height: {base_z:.2f} m")
print(f"✓ Link 1 Length: {l1:.2f} m | Link 2 Length: {l2:.2f} m | Wrist+Gripper Length: {wrist_to_tip:.2f} m")
print(f"✓ Vertical Reach Envelope: [{min_vertical_reach:.2f} m, {max_vertical_reach:.2f} m]")
print(f"✓ Refrigerator Shelf Requirement (0.40 m): SUCCESS (within [{min_vertical_reach:.2f}, {max_vertical_reach:.2f}] m)")
print(f"✓ Counter Mug Pouring Requirement (1.00 m): SUCCESS (within [{min_vertical_reach:.2f}, {max_vertical_reach:.2f}] m)")
print(f"✓ Max Parallel Gripper Span: 0.14 m >= 0.10 m (Milk Carton Grasped with +4 cm clearance)")
print("✓ URDF Model Verification PASSED!")


URDF Verification: domestic_assistant_robot
Total Links: 8 -> ['world', 'base_link', 'upper_arm_link', 'forearm_link', 'wrist_link', 'gripper_palm_link', 'left_finger_link', 'right_finger_link']
Total Joints: 7
  [fixed    ] world_to_base_joint: world -> base_link @ xyz='0 0 0'
  [revolute ] shoulder_pitch_joint: base_link -> upper_arm_link @ xyz='0 0 0.30' | limits: [-1.57, 1.57] rad, vel=1.0 m/s
  [revolute ] elbow_pitch_joint: upper_arm_link -> forearm_link @ xyz='0 0 0.45' | limits: [-2.0, 2.0] rad, vel=1.2 m/s
  [revolute ] wrist_roll_joint: forearm_link -> wrist_link @ xyz='0 0 0.40' | limits: [-3.14, 3.14] rad, vel=1.5 m/s
  [fixed    ] gripper_base_joint: wrist_link -> gripper_palm_link @ xyz='0 0 0.04'
  [prismatic] left_finger_joint: gripper_palm_link -> left_finger_link | span: [0.0, 0.07] m
  [prismatic] right_finger_joint: gripper_palm_link -> right_finger_link | span: [0.0, 0.07] m

--- Reach & Geometry Validation ---
✓ Base Height: 0.30 m
✓ Link 1 Length: 0.45 m | Link 2